# Middleware

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("OPENROUTER_API_KEY"), "Missing OPENAI_API_KEY -- check your .env file"
print("Environment OK.")

Environment OK.


In [17]:
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver 
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware,HumanInTheLoopMiddleware,ModelCallLimitMiddleware,PIIMiddleware,ToolCallRequest
from langgraph.types import Command
from langgraph.runtime import Runtime
from rich import print

In [3]:
model = init_chat_model(
    "openai/gpt-3.5-turbo",
    model_provider="openrouter",
)

In [4]:
response = model.invoke("Why do parrots talk? give me in 10 words")

In [5]:
response

AIMessage(content='Imitate sounds, communicate, show off, bond, entertain owners, socialize.', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo', 'id': 'gen-1789233003-1aC8kxOYUrD55Rk5XgFQ', 'created': 1789233003, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 3.5e-05, 'cost_details': {'upstream_inference_completions_cost': 2.55e-05, 'upstream_inference_prompt_cost': 9.5e-06, 'upstream_inference_cost': 3.5e-05}}, id='lc_run--01a09698-f87f-7043-ab26-e5153d270100-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 17, 'total_tokens': 36, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})

In [3]:
# @title
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    ToolMessage,
    SystemMessage,
)


def pretty_print_chat(state):
    """
    Pretty print a LangChain agent state.

    Args:
        state: Dictionary returned by agent.invoke(...)
    """

    role_icons = {
        HumanMessage: "🧑 Human",
        AIMessage: "🤖 AI",
        ToolMessage: "🛠️ Tool",
        SystemMessage: "⚙️ System",
    }

    print("=" * 100)

    for i, message in enumerate(state["messages"], start=1):

        role = role_icons.get(type(message), type(message).__name__)

        print(f"\n{role} ({i})")
        print("-" * 100)

        if message.content:
            print(message.content)

        # Print tool calls made by the AI
        if isinstance(message, AIMessage) and message.tool_calls:
            print("\n🔧 Tool Calls:")
            for tool in message.tool_calls:
                print(f"   • Tool : {tool['name']}")
                print(f"     Args : {tool['args']}")
                print(f"     ID   : {tool['id']}")

        # Print tool outputs
        if isinstance(message, ToolMessage):
            print(f"\nTool Name    : {message.name}")
            print(f"Tool Call ID : {message.tool_call_id}")

    print("\n" + "=" * 100)

In [7]:
@tool
def check_showtimes(name:str) -> str:
    """Fucntion is built to give showtime for a movie
    Paramaters: name -> this is movie name"""
    showing = {
        "interstellar" : "everyday 7 PM",
        "basha": "Monday , Wednesday evening 5 PM",
        "nayagan": "everyday morning 10 AM"
    }
    return showing.get(name.lower(), "No shows available")

@tool
def cancel_booking(name:str) -> str:
    """"Built to cancel booking
    Paramaters: name -> this is movie name"""
    return f"Your seat for {name} is cancelled"

@tool 
def enquire_seat(name:str) -> str:
    """Built to provide list seat availability for movie enquired
    Paramaters: name -> this is movie name"""
    return f"5 seats available for movie {name}"

@tool
def cancel_policy() ->str:
    """This provides insighs about cancellation policy after booking movie tickets"""
    cancellation = """Customers can cancel movie tickets up to 2 hours before the showtime for a full refund.
Cancellations made within 2 hours of the showtime are not eligible for refunds.
Refunds, if applicable, will be processed to the original payment method within 5–7 business days."""
    return cancellation

In [7]:
saver = InMemorySaver()
thread_config = {"configurable": {"thread_id": "middle-demo-1"}}
agent = create_agent(model="openrouter:openai/gpt-3.5-turbo", 
                     tools = [check_showtimes,cancel_booking,cancel_policy,enquire_seat],
                     checkpointer=saver,
                     )

In [8]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "I want to book 3 tickets, can you check showtim for Basha movie also check is there any seat available to book?"}]
}, config=thread_config
)

In [9]:
response

{'messages': [HumanMessage(content='I want to book 3 tickets, can you check showtim for Basha movie also check is there any seat available to book?', additional_kwargs={}, response_metadata={}, id='2e0f46f4-5d8b-48d4-a986-6e1f6860db1f'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo', 'id': 'gen-1789078126-J4YLUsTTuSVGkJkzjk4o', 'created': 1789078126, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.000162, 'cost_details': {'upstream_inference_completions_cost': 7.05e-05, 'upstream_inference_prompt_cost': 9.15e-05, 'upstream_inference_cost': 0.000162}}, id='lc_run--01a08d5d-b96c-7553-a9f9-b3971ecf64c3-0', tool_calls=[{'name': 'check_showtimes', 'args': {'name': 'Basha'}, 'id': 'call_xnA9XD0PE3tVCdw4tcwCQatv', 'type': 'tool_call'}, {'name': 'enquire_seat', 'args': {'name': 'Basha'}, 'id': 'call_YSacyVSuGr48dXkh8KgBIfFm', 'type': 'tool_call'}], invalid_tool_calls=

In [10]:
agent = create_agent(model="openrouter:openai/gpt-3.5-turbo", 
                     tools = [check_showtimes,cancel_booking,cancel_policy,enquire_seat],
                     checkpointer=saver,
                     middleware=[
                         SummarizationMiddleware(
                             model= "openrouter:deepseek/deepseek-v4-flash-0731",
                             trigger=('tokens', 100),
                             keep=("messages", 2)
                         ),
                     ],
                     )

In [11]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "I want to book 3 tickets, can you check showtim for Basha movie also check is there any seat available to book?"}]
},config=thread_config
)

In [12]:
response

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nUser wants to book 3 tickets for the movie "Basha" and asked to check showtimes and seat availability.\n\n## SUMMARY\n\n- User requested booking 3 tickets for "Basha".\n- Checked showtimes for "Basha": available on Monday and Wednesday evening at 5 PM.\n- Checked seat availability: 5 seats available for "Basha".\n- No booking has been made yet; the user has not selected a specific showtime or confirmed the booking.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\n- Ask the user which showtime they prefer (Monday or Wednesday, 5 PM) to proceed with booking 3 tickets.\n- Confirm seat selection/booking once the showtime is chosen.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='e733f525-ffd7-4e7d-8688-8e78a92a9844'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo', 'id': 'gen-1789078156-HnL0EOLXtWBTQ9IzNZ

In [13]:
response['messages'][-1].response_metadata

{'model_name': 'openai/gpt-3.5-turbo',
 'id': 'gen-1789078166-LBvLzi9klXC3DAwU8wnf',
 'created': 1789078166,
 'object': 'chat.completion',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'openrouter',
 'cost': 0.0002835,
 'cost_details': {'upstream_inference_completions_cost': 8.7e-05,
  'upstream_inference_prompt_cost': 0.0001965,
  'upstream_inference_cost': 0.0002835}}

In [14]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "can you check was basha has showtime in Tuesday? let me know what is the cancellation policy?"}]
},config=thread_config
)

In [15]:
response

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nUser wants to book 3 tickets for the movie "Basha." They previously requested showtime and seat availability, and now are asking to check whether "Basha" has a showtime on Tuesday and to know the cancellation policy.\n\n## SUMMARY\n\n- User requested booking 3 tickets for "Basha."\n- Confirmed showtimes: Monday and Wednesday evening at 5 PM. Tuesday was not previously mentioned.\n- Seat availability: 5 seats available for "Basha."\n- No booking has been made yet; the user has not selected a showtime or confirmed booking.\n- Current user request: Check if "Basha" has a Tuesday showtime, and provide the cancellation policy.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\n- Check whether "Basha" has a showtime on Tuesday.\n- Check and provide the cancellation policy for the booking.\n- After presenting Tuesday options/cancellation policy, ask the user which showtime they prefer (Monday,

In [16]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "what are the movies has shows? on what days?"}]
},config=thread_config
)

In [17]:
response

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user originally wanted to book 3 tickets for the movie "Basha" and asked about Tuesday showtimes and cancellation policy. The user has now asked a new question: "what are the movies has shows? on what days?" — likely wanting a list of available movies and their showtimes.\n\n## SUMMARY\n\n- "Basha" showtimes: Monday and Wednesday at 5 PM only. There is NO Tuesday showtime.\n- Seat availability for "Basha": 5 seats available.\n- Cancellation policy for movie tickets:\n  - Cancel up to 2 hours before showtime for a full refund.\n  - Cancellations within 2 hours of showtime are not refundable.\n  - Refunds are processed to original payment method within 5–7 business days.\n- No booking has been made yet; the user has not selected a showtime or confirmed booking.\n- The user was previously asked to choose between Monday or Wednesday 5 PM for "Basha."\n- The user has now shifted to

In [18]:
pretty_print_chat(response)


🧑 Human (1)
----------------------------------------------------------------------------------------------------
Here is a summary of the conversation to date:

## SESSION INTENT

The user originally wanted to book 3 tickets for the movie "Basha" and asked about Tuesday showtimes and cancellation policy. The user has now asked a new question: "what are the movies has shows? on what days?" — likely wanting a list of available movies and their showtimes.

## SUMMARY

- "Basha" showtimes: Monday and Wednesday at 5 PM only. There is NO Tuesday showtime.
- Seat availability for "Basha": 5 seats available.
- Cancellation policy for movie tickets:
  - Cancel up to 2 hours before showtime for a full refund.
  - Cancellations within 2 hours of showtime are not refundable.
  - Refunds are processed to original payment method within 5–7 business days.
- No booking has been made yet; the user has not selected a showtime or confirmed booking.
- The user was previously asked to choose between Monda

# Human in the Loop(HITL)

In [19]:
agent = create_agent(model="openrouter:openai/gpt-3.5-turbo", 
                     tools = [check_showtimes,cancel_booking,cancel_policy,enquire_seat],
                     checkpointer=saver,
                     middleware=[
                         HumanInTheLoopMiddleware(
                             interrupt_on={
                                 "cancel_booking":True,
                             },
                             description_prefix="Tool execution pending approval",
                         ),
                     ],
                     )

In [20]:
hitl_config = {"configurable": {"thread_id": "middle-hitldemo-1"}}

In [21]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "can you cancel the booking A123"}]
},config=hitl_config
)

In [22]:
response

{'messages': [HumanMessage(content='can you cancel the booking A123', additional_kwargs={}, response_metadata={}, id='b81621d6-8caf-4d8f-924b-a4780fdce9d2'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo', 'id': 'gen-1789078314-1hfbK8MedVDZQpUtm0Hw', 'created': 1789078314, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.000104, 'cost_details': {'upstream_inference_completions_cost': 2.25e-05, 'upstream_inference_prompt_cost': 8.15e-05, 'upstream_inference_cost': 0.000104}}, id='lc_run--01a08d60-9919-7943-bb63-fab8829976f0-0', tool_calls=[{'name': 'cancel_booking', 'args': {'name': 'A123'}, 'id': 'call_2Qt3Mfg4zviFi7V4aGqRXy7V', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 163, 'output_tokens': 15, 'total_tokens': 178, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})],
 '__

## Resume with approval decision

In [23]:
agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=hitl_config
)

{'messages': [HumanMessage(content='can you cancel the booking A123', additional_kwargs={}, response_metadata={}, id='b81621d6-8caf-4d8f-924b-a4780fdce9d2'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo', 'id': 'gen-1789078314-1hfbK8MedVDZQpUtm0Hw', 'created': 1789078314, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.000104, 'cost_details': {'upstream_inference_completions_cost': 2.25e-05, 'upstream_inference_prompt_cost': 8.15e-05, 'upstream_inference_cost': 0.000104}}, id='lc_run--01a08d60-9919-7943-bb63-fab8829976f0-0', tool_calls=[{'name': 'cancel_booking', 'args': {'name': 'A123'}, 'id': 'call_2Qt3Mfg4zviFi7V4aGqRXy7V', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 163, 'output_tokens': 15, 'total_tokens': 178, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}),
  Too

In [24]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "can you cancel the booking A123"}]
},config=hitl_config
)

In [ ]:
agent.get_state(hitl_config).next

('HumanInTheLoopMiddleware.after_model',)

: 

# HITL based on condition 

In [20]:
@tool
def refund_money(amount:int) -> str:
    "Refund the amount requested in this function"
    print(f"Amount requested is {amount}")
    return f"Amount {amount} has been refrunded"

In [38]:
def is_refund_approval(request: ToolCallRequest)->bool:
    amount = request.tool_call["args"].get("amount",0)
    print(amount)
    return True if amount > 100 else False

    # print("REQUEST:")
    # print(request)
    # print("TOOL CALL:")
    # print(request.tool_call)
    # print("ARGS:")
    # print(request.tool_call["args"])



In [39]:
saver = InMemorySaver()
hitl_cond_config = {"configurable": {"thread_id": "HITL_COND_2"}}
agent = create_agent(
    model="openrouter:openai/gpt-3.5-turbo",
    tools=[refund_money],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "refund_money":{                    
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "when": is_refund_approval,
                },
            },
        ),
    ],
    checkpointer=saver
)

In [42]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "can you process refund for 150 usd"}]
}, config=hitl_cond_config
)

150

In [43]:
print(response)

{
    'messages': [
        HumanMessage(
            content='can you process refund for 50 usd',
            additional_kwargs={},
            response_metadata={},
            id='3fce8a34-aa77-4632-85ed-3580586f1dca'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-3.5-turbo',
                'id': 'gen-1789841926-yGZdILSAiXX31YRSB5U1',
                'created': 1789841926,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 5.05e-05,
                'cost_details': {
                    'upstream_inference_completions_cost': 2.25e-05,
                    'upstream_inference_prompt_cost': 2.8e-05,
                    'upstream_inference_cost': 5.05e-05
                }
            },
            id='lc_run--01a0bae4-63ff-7c93-99f2-c022bce7dce9-0',
            tool_calls=[
                {
                    'name': 'refund_money',
                    'args': {'amount': 50},
                    'id': 'call_yZjplKMa6eJe8CuOdJ0eY4P1',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 56,
                'output_tokens': 15,
                'total_tokens': 71,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Amount 50 has been refrunded',
            name='refund_money',
            id='31f345fd-b691-48da-9cd5-87f81000364a',
            tool_call_id='call_yZjplKMa6eJe8CuOdJ0eY4P1'
        ),
        AIMessage(
            content='The refund for 50 USD has been processed successfully.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-3.5-turbo',
                'id': 'gen-1789841927-nBWSWcXDbXBUfK0RgdDM',
                'created': 1789841927,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 6.1e-05,
                'cost_details': {
                    'upstream_inference_completions_cost': 1.8e-05,
                    'upstream_inference_prompt_cost': 4.3e-05,
                    'upstream_inference_cost': 6.1e-05
                }
            },
            id='lc_run--01a0bae4-67d9-7250-8ae1-99344352ac53-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 86,
                'output_tokens': 12,
                'total_tokens': 98,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        HumanMessage(
            content='can you process refund for 150 usd',
            additional_kwargs={},
            response_metadata={},
            id='46c58b48-0160-4b61-8738-a0ada146f408'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-3.5-turbo',
                'id': 'gen-1789841941-VbqiTplPwgnmSWnU8Gkv',
                'created': 1789841941,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 7.95e-05,
                'cost_details': {
                    'upstream_inference_completions_cost': 2.25e-05,
                    'upstream_inference_prompt_cost': 5.7e-05,
                    'upstream_inference_cost': 7.95e-05
                }
            },
            id=

In [24]:
saver = InMemorySaver()
thread_config = {"configurable": {"thread_id": "model-demo-1"}}
agent = create_agent(
    model="openrouter:openai/gpt-3.5-turbo",
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools = [check_showtimes,cancel_booking,cancel_policy,enquire_seat],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=5,
            run_limit=3,
            exit_behavior='error'
        ),
    ],
)

In [25]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "can you check showtime for Basha ? enquire seat avability and explain about cancellation policy?"}]
}, config=thread_config
)

In [26]:
print(response)

{
    'messages': [
        HumanMessage(
            content='can you check showtime for Basha ? enquire seat avability and explain about cancellation 
policy?',
            additional_kwargs={},
            response_metadata={},
            id='c4eefd94-46c1-4050-8d76-1beb513016d9'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-3.5-turbo',
                'id': 'gen-1789233464-Za7RnJqGl1fZ1lyq2Srm',
                'created': 1789233464,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0001765,
                'cost_details': {
                    'upstream_inference_completions_cost': 8.85e-05,
                    'upstream_inference_prompt_cost': 8.8e-05,
                    'upstream_inference_cost': 0.0001765
                }
            },
            id='lc_run--01a096a0-0123-77b0-9418-e42f15602a8e-0',
            tool_calls=[
                {
                    'name': 'check_showtimes',
                    'args': {'name': 'Basha'},
                    'id': 'call_RpYcxVWbK6oZQO03g33HAiqw',
                    'type': 'tool_call'
                },
                {
                    'name': 'enquire_seat',
                    'args': {'name': 'Basha'},
                    'id': 'call_4OG1cs8NIX7TDx5GtJEnRxjk',
                    'type': 'tool_call'
                },
                {'name': 'cancel_policy', 'args': {}, 'id': 'call_w3jQuhTqaUD7ptBm4DEUCsJV', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 176,
                'output_tokens': 59,
                'total_tokens': 235,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Monday , Wednesday evening 5 PM',
            name='check_showtimes',
            id='d3fe0dce-ee84-4aed-bb98-5047eeafd614',
            tool_call_id='call_RpYcxVWbK6oZQO03g33HAiqw'
        ),
        ToolMessage(
            content='5 seats available for movie Basha',
            name='enquire_seat',
            id='e5db7c2d-a049-4907-a3cd-bfeecdcf2978',
            tool_call_id='call_4OG1cs8NIX7TDx5GtJEnRxjk'
        ),
        ToolMessage(
            content='Customers can cancel movie tickets up to 2 hours before the showtime for a full 
refund.\nCancellations made within 2 hours of the showtime are not eligible for refunds.\nRefunds, if applicable, 
will be processed to the original payment method within 5–7 business days.',
            name='cancel_policy',
            id='bbfbc384-6ff3-4b43-8702-486086063120',
            tool_call_id='call_w3jQuhTqaUD7ptBm4DEUCsJV'
        ),
        AIMessage(
            content='The showtime for the movie "Basha" is on Monday and Wednesday evening at 5 PM. There are 5 
seats available for the movie "Basha". \n\nRegarding the cancellation policy:\nCustomers can cancel movie tickets 
up to 2 hours before the showtime for a full refund.\nCancellations made within 2 hours of the showtime are not 
eligible for refunds.\nRefunds, if applicable, will be processed to the original payment method within 5–7 business
days.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-3.5-turbo',
                'id': 'gen-1789233465-9wb9AcLCZbQHxZP4lAIg',
                'created': 1789233465,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0003125,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.00015,
              

# PII - Personal Identifiable Information

In [29]:
saver = InMemorySaver()
thread_config = {"configurable": {"thread_id": "pii-demo-1"}}
agent = create_agent(
    model="openrouter:openai/gpt-3.5-turbo",
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools = [check_showtimes,cancel_booking,cancel_policy,enquire_seat],
    middleware=[
        PIIMiddleware('email',strategy='mask',apply_to_input=True,apply_to_output=True)
    ],
)

In [30]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "here is my email test@gmail.com to book tickets"}]
}, config=thread_config
)

In [31]:
print(response)

{
    'messages': [
        HumanMessage(
            content='here is my email test@****.com to book tickets',
            additional_kwargs={},
            response_metadata={},
            id='55a2f310-688e-4682-971c-73043a6689a8'
        ),
        AIMessage(
            content='Hello! How can I assist you with booking tickets using your email address test@****.com? Let 
me know the name of the movie you would like to book tickets for.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-3.5-turbo',
                'id': 'gen-1789234449-JaucRuQwtFajZOthMNci',
                'created': 1789234449,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0001395,
                'cost_details': {
                    'upstream_inference_completions_cost': 5.55e-05,
                    'upstream_inference_prompt_cost': 8.4e-05,
                    'upstream_inference_cost': 0.0001395
                }
            },
            id='lc_run--01a096af-0bcd-73f3-85db-8d29b8f3a961-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 168,
                'output_tokens': 37,
                'total_tokens': 205,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        )
    ]
}

In [32]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "what is my email?"}]
}, config=thread_config
)

In [33]:
print(response)

{
    'messages': [
        HumanMessage(
            content='here is my email test@****.com to book tickets',
            additional_kwargs={},
            response_metadata={},
            id='55a2f310-688e-4682-971c-73043a6689a8'
        ),
        AIMessage(
            content='Hello! How can I assist you with booking tickets using your email address test@****.com? Let 
me know the name of the movie you would like to book tickets for.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-3.5-turbo',
                'id': 'gen-1789234449-JaucRuQwtFajZOthMNci',
                'created': 1789234449,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0001395,
                'cost_details': {
                    'upstream_inference_completions_cost': 5.55e-05,
                    'upstream_inference_prompt_cost': 8.4e-05,
                    'upstream_inference_cost': 0.0001395
                }
            },
            id='lc_run--01a096af-0bcd-73f3-85db-8d29b8f3a961-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 168,
                'output_tokens': 37,
                'total_tokens': 205,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        HumanMessage(
            content='what is my email?',
            additional_kwargs={},
            response_metadata={},
            id='688a2714-ec79-4b4c-8115-72a8df9bb80c'
        ),
        AIMessage(
            content='Your email is test@****.com. How may I assist you further? Please provide the name of the 
movie you would like to book tickets for.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-3.5-turbo',
                'id': 'gen-1789234480-OdmujJF5X9S4DifOYcvO',
                'created': 1789234480,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0001565,
                'cost_details': {
                    'upstream_inference_completions_cost': 4.8e-05,
                    'upstream_inference_prompt_cost': 0.0001085,
                    'upstream_inference_cost': 0.0001565
                }
            },
            id='lc_run--01a096af-8338-7172-8e67-f45f8c81c324-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 217,
                'output_tokens': 32,
                'total_tokens': 249,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        )
    ]
}

In [35]:
response['messages'][-1]

AIMessage(content='Your email is test@****.com. How may I assist you further? Please provide the name of the movie you would like to book tickets for.', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-3.5-turbo', 'id': 'gen-1789234480-OdmujJF5X9S4DifOYcvO', 'created': 1789234480, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0001565, 'cost_details': {'upstream_inference_completions_cost': 4.8e-05, 'upstream_inference_prompt_cost': 0.0001085, 'upstream_inference_cost': 0.0001565}}, id='lc_run--01a096af-8338-7172-8e67-f45f8c81c324-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 217, 'output_tokens': 32, 'total_tokens': 249, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})